# Execution Plan Analysis: Reading Physical Operators and Bottlenecks

**Module**: 10_performance_and_optimization  
**Notebook**: 02_explain_and_execution_plans.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Deep reading of physical plans: Scan, Filter, Project, Exchange, Sort, HashAggregate, and join operators.
- **Why does this topic exist?**: Before touching cluster configurations, you must verify the query plan is optimal.
- **What problem does it solve?**: Missing that a query is triggering an unintended Cartesian product or full scan.
- **Where is it used in real-world Data Engineering?**: Authoring code reviews for high-throughput batch ETL pipelines.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Plan_Analysis_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
q = orders.filter("quantity > 3").groupBy("channel").agg(F.count("order_id").alias("cnt"))
q.explain("formatted")

---

## Experiment: Detecting Exchange Nodes in Complex Queries

- **1. What are we testing?**: Count shuffle exchange points in multi-step transformations.
- **2. What do we expect?**: Each groupBy, distinct, and join creates an Exchange operator.

In [ ]:
plan_text = q._jdf.queryExecution().executedPlan().toString()
print("Total Exchange (Shuffle) nodes:", plan_text.count("Exchange"))

**3. What happened?**: Counted physical Exchange nodes.  
**4. Why did it happen?**: Exchange operators represent network shuffle boundaries.  
**5. Practical Engineering Lesson**: Fewer Exchange nodes almost always means faster query execution.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify HashAggregate
**Problem Statement**: Find HashAggregate in explain output.  
**Input / Context**: q

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): BroadcastExchange Node
**Problem Statement**: Verify BroadcastExchange presence when broadcasting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Cost Model Statistics
**Problem Statement**: Explain Statistics output in explain('cost').  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify HashAggregate):
```python
print('HashAggregate is visible in formatted plan.')
```

#### Solution 2 (BroadcastExchange Node):
```python
orders.join(F.broadcast(orders.limit(5)),'order_id').explain()
```

#### Solution 3 (Cost Model Statistics):
```python
print('Statistics show estimated row count and byte size at each node.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Plan Verification**: Verify filter pushdown, column pruning, and exchange count.
- **Formatted Plan**: Use explain('formatted') for clear node breakdowns.

### Essential Technical Terms
- `Exchange`: Shuffle operator.
- `HashAggregate`: In-memory hash aggregation.

### Common Engineering Mistakes
- **Running queries without checking plans**: Leads to surprise shuffles in production.

### Interview Questions on This Topic
- **Q: How do you know if an operator participates in WholeStageCodegen?**
  - *A*: It is prefixed with an asterisk (*) in the physical plan.

### What to Study Next
Open 03_query_optimization.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')